# Практична робота 1

## Формулювання задачі та первинний аудит даних

**Набір даних:** Iris (Ірис) — індивідуальний варіант, **цільова ознака: `sepal width` (ширина чашолистка)**

**Дисципліна:** Машинне навчання

**Виконав(ла):** _(ПІБ студента)_

**Група:** _(вказати групу)_

---

> **Важливо.** Метою цієї практичної роботи є **розуміння та підготовка даних**, а не побудова моделі. Модель у цій роботі не будується — тут лише формулюється задача, описується набір даних і проводиться його первинний аудит.

## 1. Формулювання задачі та опис індивідуального набору даних

### 1.1. Назва та джерело набору даних

Використовується класичний набір даних **Iris (Ірис Фішера)**, зібраний Р. Фішером у 1936 р. У цій роботі він завантажується через вбудований завантажувач `sklearn.datasets.load_iris`, який містить точну копію оригінального набору (UCI Machine Learning Repository, "Iris Data Set", https://archive.ics.uci.edu/dataset/53/iris).

### 1.2. Предметна область

Предметна область — **ботаніка / морфометрія рослин**. Набір містить результати вимірювання квіток трьох видів ірису (*Iris setosa*, *Iris versicolor*, *Iris virginica*): довжину та ширину чашолистків (sepal) і пелюсток (petal).

### 1.3. Об'єкти, що описуються рядками таблиці

Кожен рядок таблиці відповідає **одній окремій квітці ірису** (одному ботанічному зразку), для якої виміряно чотири морфометричні ознаки та визначено вид рослини.

### 1.4–1.7. Кількість об'єктів і ознак, цільова змінна, тип задачі, практичний зміст

Ці пункти конкретизуються нижче, після завантаження даних (кількість об'єктів/ознак наведена за фактичним результатом `df.shape`), а також у підсумку розділу 2.

- **Цільова змінна (за умовою індивідуального варіанта):** `sepal_width` (ширина чашолистка, см) — неперервна кількісна ознака.
- **Тип майбутньої задачі машинного навчання:** **регресія** (прогнозування неперервного числового значення), а не класифікація, як у класичному використанні цього набору даних (де ціллю зазвичай є вид рослини `species`).
- **Ознаки-предиктори:** `sepal_length`, `petal_length`, `petal_width`, а також категоріальна ознака `species` (вид рослини).
- **Практичний зміст майбутнього прогнозування:** у природних умовах виміряти довжину чашолистка, довжину та ширину пелюстки і визначити вид квітки зазвичай простіше й швидше, ніж точно виміряти ширину чашолистка (наприклад, через її нерегулярну форму або пошкодження зразка). Модель, що прогнозує `sepal_width` за іншими ознаками, дозволила б **оцінювати цю характеристику непрямим шляхом**, коли пряме вимірювання ускладнене, відсутнє або дороге, а також слугує навчальним прикладом задачі регресії на добре вивчених, чистих даних.

## 2. Первинний аудит даних

Завантажимо дані та проведемо їх первинний аудит: розмірність, типи стовпців, кількість унікальних значень, пропуски, дублікати та потенційні службові/ідентифікаційні ознаки.

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris

pd.set_option('display.max_columns', None)

# Завантаження даних
iris = load_iris(as_frame=True)
df = iris.frame.copy()

# Перейменування стовпців у зручний вигляд snake_case
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'target']

# Розшифровка кодів виду рослини (0/1/2) у назви видів
species_map = dict(zip(range(3), iris.target_names))
df['species'] = df['target'].map(species_map)
df = df.drop(columns=['target'])

df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


Перші рядки показують структуру таблиці: чотири числові морфометричні ознаки (в сантиметрах) та одну категоріальну ознаку `species` (вид рослини).

In [2]:
df.shape

(150, 5)

**Розмірність таблиці:** `(150, 5)` — набір містить **150 об'єктів (квіток)** та **5 стовпців**: 4 числові ознаки (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`) та 1 категоріальна ознака (`species`). Оскільки цільовою змінною за умовою варіанта є `sepal_width`, реальна кількість ознак-предикторів для майбутньої регресії — **4** (`sepal_length`, `petal_length`, `petal_width`, `species`).

In [3]:
list(df.columns)

['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


`df.info()` підтверджує: пропуски відсутні (усі стовпці мають 150 непорожніх значень із 150), 4 стовпці мають тип `float64`, стовпець `species` — рядковий/об'єктний тип.

In [5]:
df.dtypes

sepal_length    float64
sepal_width     float64
petal_length    float64
petal_width     float64
species             str
dtype: object

**Формальні типи даних:** `sepal_length`, `sepal_width`, `petal_length`, `petal_width` — `float64` (неперервні числові ознаки); `species` — текстовий/категоріальний тип. Формальний тип узгоджується зі змістовним: усі вимірювальні ознаки дійсно є неперервними, `species` — це категорія.

In [6]:
df.nunique()

sepal_length    35
sepal_width     23
petal_length    43
petal_width     22
species          3
dtype: int64

**Кількість унікальних значень:** `sepal_length` — 35, `sepal_width` — 23, `petal_length` — 43, `petal_width` — 22, `species` — 3. Числові ознаки мають помірну кількість унікальних значень (набір даних невеликий і виміряний з обмеженою точністю, тому значення повторюються), що очікувано для фізичних вимірювань. `species` — категоріальна ознака рівно з 3 значеннями.

In [7]:
df['species'].value_counts()

species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64

Класи за видом рослини **збалансовані**: по 50 об'єктів кожного з трьох видів (`setosa`, `versicolor`, `virginica`). Це корисно знати навіть у задачі регресії, оскільки `species` планується використовувати як категоріальний предиктор.

In [8]:
missing_count = df.isnull().sum()
missing_share = (df.isnull().mean() * 100).round(2)
pd.DataFrame({'кількість_пропусків': missing_count, 'частка_пропусків_%': missing_share})

,кількість_пропусків,частка_пропусків_%
sepal_length,0,0.0
sepal_width,0,0.0
petal_length,0,0.0
petal_width,0,0.0
species,0,0.0


**Пропущені значення відсутні** — у жодному стовпці немає пропусків (0 пропусків, 0.0% у кожному стовпці). Це очікувано для добре відомого еталонного набору даних.

In [9]:
duplicate_count = df.duplicated().sum()
print('Кількість повних дублікатів рядків:', duplicate_count)
df[df.duplicated(keep=False)].sort_values(list(df.columns))

Кількість повних дублікатів рядків: 1


,sepal_length,sepal_width,petal_length,petal_width,species
101,5.8,2.7,5.1,1.9,virginica
142,5.8,2.7,5.1,1.9,virginica


У наборі даних виявлено **1 повний дублікат** — рядки з індексами 101 і 142 повністю збігаються за всіма ознаками (`sepal_length=5.8, sepal_width=2.7, petal_length=5.1, petal_width=1.9, species=virginica`). Це відома особливість класичного набору Iris. Дублікат варто мати на увазі на етапі підготовки даних до моделювання (наприклад, розглянути його видалення, щоб уникнути витоку інформації між навчальною та тестовою вибіркою), хоча в межах цієї роботи модель ще не будується.

In [10]:
df.describe()

,sepal_length,sepal_width,petal_length,petal_width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


Додатково: описова статистика (`df.describe()`) показує адекватні, фізично правдоподібні діапазони значень (наприклад, `sepal_width` від 2.0 до 4.4 см) без явних викидів чи помилок введення (від'ємних значень, аномально великих чисел тощо).

### Потенційні службові або ідентифікаційні ознаки

- Явного стовпця-ідентифікатора (`id`, `index` тощо) у наборі даних **немає** — вбудований завантажувач `sklearn` не додає технічних службових стовпців.
- Стовпець `species` **не є службовим**, це змістовна категоріальна ознака (вид рослини). У класичній постановці задачі (класифікація) саме він відігравав роль цільової змінної; у поточному індивідуальному варіанті (регресія, ціль — `sepal_width`) він переходить у розряд ознаки-предиктора.
- Індекс `pandas` (0–149) є лише технічним порядковим номером рядка і не несе змістовної інформації — його не слід використовувати як ознаку.

## 3. Підсумок первинного аудиту

| Перевірка | Результат |
|---|---|
| Розмірність | 150 рядків × 5 стовпців |
| Типи стовпців | 4 × `float64` (числові), 1 × текстовий (`species`) |
| Унікальні значення | 35 / 23 / 43 / 22 / 3 (по стовпцях) |
| Пропущені значення | 0 у кожному стовпці (0.0%) |
| Повні дублікати рядків | 1 (рядки 101 та 142) |
| Службові/ID-ознаки | Відсутні; `species` — змістовна категоріальна ознака, а не службова |
| Баланс класів `species` | Збалансовано: 50 / 50 / 50 |

**Виявлені потенційні проблеми:**
1. Один повний дублікат рядка — потребує рішення на етапі підготовки даних (видаляти чи залишати).
2. Ознака `species` категоріальна — для регресійних моделей, чутливих до типу входу, її потрібно буде закодувати (наприклад, one-hot encoding) на наступних етапах роботи.
3. Пропущені значення відсутні, набір даних чистий і повний — додаткової обробки пропусків не потрібно.

Ці спостереження буде враховано на наступних етапах роботи (розвідувальний аналіз, підготовка ознак, побудова моделі регресії для прогнозування `sepal_width`), які виходять за межі поточної практичної роботи.